# LoRA vs LoRA+ Experimental Evaluation

## RoBERTa-base + QQP

### 1. Project Setup and GitHub Integration

Clone the LoRA-Plus project repository from GitHub into Google Colab to utilize GPU resources for model training.

Experiment results are saved in the repository's `results/` directory and can be committed and pushed to GitHub for persistent storage.

In [1]:
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = "https://github.com/Ramprasath-Meganathan/LoRA-Plus.git"
PROJECT_DIR = Path("/content/LoRA-Plus")

if not PROJECT_DIR.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(PROJECT_DIR)],
        check=True
    )
else:
    print("Repository already exists. Using existing files.")

for filename in [
    "config.py",
    "model.py",
    "utils.py",
    "train.py",
    "evaluate_results.py"
]:
    assert (PROJECT_DIR / filename).is_file(), (
        f"Missing project file: {filename}"
    )

os.chdir(PROJECT_DIR)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Project:", os.getcwd())

Project: /content/LoRA-Plus


## 2. Install Required Dependencies

Install the required libraries from the repository's `requirements.txt` file for model fine-tuning, LoRA/LoRA+ optimization, dataset processing, and evaluation.

Remove incompatible `torchao` installations to ensure compatibility with PEFT in Google Colab.

In [2]:
%pip install -r requirements.txt
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


## 3. Experiment Configuration and GPU Verification

Select RoBERTa-base and the Quora Question Pairs (QQP) dataset.

The experiment uses three training epochs, nine hyperparameter screening configurations, and three random seeds (42, 43, 44) for final evaluation.

Verify that a GPU is available before starting training.

In [3]:
import torch

MODEL_NAME = "roberta"
DATASET_NAME = "qqp"

assert torch.cuda.is_available(), (
    "GPU unavailable. Enable a GPU runtime in Colab."
)

print("Model:", MODEL_NAME)
print("Dataset:", DATASET_NAME)
print("GPU:", torch.cuda.get_device_name(0))

Model: roberta
Dataset: qqp
GPU: Tesla T4


## 4. Import Shared Project Modules

Import the reusable configuration, dataset preparation, training, and evaluation functions from the project's Python modules.

These modules provide a consistent experimental pipeline for standard LoRA and LoRA+.

In [ ]:
from config import (
    COMMON_CONFIG,
    DATASET_CONFIGS,
    LORA_LEARNING_RATES,
    LORAPLUS_CONFIGS,
    get_output_dir
)

from utils import prepare_dataset, track_compute, detect_gpu
from train import run_experiment

from evaluate_results import (
    select_best,
    summarize_final,
    plot_convergence
)

print("All project modules imported successfully!")
print("Results directory:", get_output_dir(MODEL_NAME, DATASET_NAME))
print("Current GPU:", detect_gpu())

assert COMMON_CONFIG["epochs"] == 3
assert COMMON_CONFIG["seeds"] == [42, 43, 44]
assert DATASET_CONFIGS["qqp"]["primary_metric"] == "f1"

All project modules imported successfully!
Results directory: /content/LoRA-Plus/results/roberta/qqp


## 5. QQP Dataset Loading and Tokenization

Load the GLUE Quora Question Pairs dataset and tokenize both questions using the RoBERTa tokenizer.

Use a fixed 50,000-example training subset and the complete validation split. Prepare the dataset once and reuse it across all training configurations.

In [ ]:
with track_compute("50-step sanity test"):

    tokenizer, train_dataset, eval_dataset, data_collator = (
        prepare_dataset(
            model_name=MODEL_NAME,
            dataset_name=DATASET_NAME,
            pilot=False
        )
    )

print("Training examples:", len(train_dataset))
print("Validation examples:", len(eval_dataset))

assert len(train_dataset) == 50000
assert len(eval_dataset) == 40430

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

... (9 outputs left out)

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

Model: roberta
Dataset: qqp
Training examples: 50000
Validation examples: 40430
Training examples: 50000
Validation examples: 40430


## 6. Training Pipeline Sanity Test

Run a short 50-step LoRA+ training experiment to verify model initialization, adapter configuration, optimizer learning rates, GPU execution, and validation metrics.

This sanity test is excluded from the official screening and final experiment results.

In [ ]:
with track_compute("50-step sanity test"):

    test = run_experiment(
        model_name=MODEL_NAME,
        dataset_name=DATASET_NAME,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        lam=4,
        seed=42,
        epochs=3,
        base_lr=1e-4,
        stage="test",
        max_steps=50
    )

print("Training steps:", test["total_steps"])
print("Validation accuracy:", test["final_acc"])
print("Validation binary F1:", test["final_f1"])

assert test["total_steps"] == 50

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 887,042 || all params: 125,534,212 || trainable%: 0.7066
A 147456 0.0001
B 147456 0.0004
other 592130 0.0001


Step,Training Loss,Validation Loss,Accuracy,F1,F1 Macro
50,0.661899,0.655608,0.631833,0.000000,0.387192


Training Loss,Validation Loss,Step,Accuracy,F1,F1 Macro
0.661899,0.655608,50,0.631833,0.000000,0.387192


Training steps: 50
Validation accuracy: 0.6318327974276527
Validation binary F1: 0.0


## 7. Standard LoRA Hyperparameter Screening

Evaluate standard LoRA using three learning rates: 1e-4, 2e-4, and 4e-4.

Both adapter matrices A and B use equal learning rates. Each configuration is trained for three epochs using random seed 42.

In [ ]:
for lr in LORA_LEARNING_RATES:

    with track_compute(f"LoRA LR={lr}"):

        print(f"\nRunning standard LoRA: LR={lr}")

        run_experiment(
            model_name=MODEL_NAME,
            dataset_name=DATASET_NAME,
            tokenizer=tokenizer,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            data_collator=data_collator,
            lam=1,
            seed=COMMON_CONFIG["screening_seed"],
            epochs=COMMON_CONFIG["epochs"],
            base_lr=lr,
            stage="screening"
        )


Running standard LoRA: LR=0.0001


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 887,042 || all params: 125,534,212 || trainable%: 0.7066
A 147456 0.0001
B 147456 0.0001
other 592130 0.0001


Step,Training Loss,Validation Loss,Accuracy,F1,F1 Macro
200,0.567575,0.519665,0.716547,0.671915,0.711203
400,0.458642,0.435900,0.786965,0.745336,0.781116
600,0.415496,0.403307,0.812367,0.762090,0.803596
800,0.415719,0.389170,0.818452,0.768702,0.809645
1000,0.412667,0.393445,0.814618,0.773872,0.808397
1200,0.401333,0.376503,0.824586,0.760922,0.811198
1400,0.435690,0.369563,0.827405,0.767756,0.815216
1600,0.399406,0.367135,0.829533,0.780103,0.820461
1800,0.399909,0.388023,0.817586,0.784111,0.813092
2000,0.386938,0.357303,0.834529,0.782989,0.824638


Training Loss,Validation Loss,Step,Accuracy,F1,F1 Macro
0.357246,0.348328,4689,0.840020,0.794784,0.831849


... (9 outputs left out)

trainable params: 887,042 || all params: 125,534,212 || trainable%: 0.7066
A 147456 0.0004
B 147456 0.0004
other 592130 0.0001


Step,Training Loss,Validation Loss,Accuracy,F1,F1 Macro
200,0.434836,0.425116,0.797304,0.762512,0.792858
400,0.419788,0.416665,0.793643,0.766727,0.790859
600,0.389176,0.387104,0.826391,0.788118,0.820536
800,0.389146,0.354268,0.837052,0.787305,0.827622
1000,0.375465,0.385106,0.819614,0.790389,0.816038
1200,0.370828,0.344151,0.843087,0.789585,0.832241
1400,0.398833,0.340531,0.844942,0.804479,0.838004
1600,0.358674,0.346587,0.843260,0.803668,0.836616
1800,0.350037,0.371825,0.831313,0.801951,0.827522
2000,0.350904,0.332808,0.849666,0.799207,0.839532


Training Loss,Validation Loss,Step,Accuracy,F1,F1 Macro
0.307209,0.317162,4689,0.858941,0.816334,0.850919


Saved result: /content/LoRA-Plus/results/roberta/qqp/runs/roberta_qqp_lam1_lr0.0004_seed42_ep3_bs32_steps-1_screening.json


## 8A. LoRA+ Hyperparameter Screening - Batch 1

Evaluate the first two LoRA+ configurations using a base learning rate of 1e-4 and learning-rate ratios of 4× and 8×.

Each configuration is trained for three epochs using random seed 42.

In [ ]:
# LoRA+ Screening — Batch 1
# Configurations 1 and 2

for lr, ratio in LORAPLUS_CONFIGS[0:2]:

    with track_compute(f"LoRA+ LR={lr}, ratio={ratio}"):

        print(f"\nRunning LoRA+: A LR={lr}, B LR={lr * ratio}, Ratio={ratio}")

        run_experiment(
            model_name=MODEL_NAME,
            dataset_name=DATASET_NAME,
            tokenizer=tokenizer,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            data_collator=data_collator,
            lam=ratio,
            seed=COMMON_CONFIG["screening_seed"],
            epochs=COMMON_CONFIG["epochs"],
            base_lr=lr,
            stage="screening"
        )

print("Batch 1 completed!")


Running LoRA+: A LR=0.0001, B LR=0.0004, Ratio=4


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 887,042 || all params: 125,534,212 || trainable%: 0.7066
A 147456 0.0001
B 147456 0.0004
other 592130 0.0001


Step,Training Loss,Validation Loss,Accuracy,F1,F1 Macro
200,0.463163,0.475521,0.771086,0.739904,0.767748
400,0.435651,0.421772,0.793866,0.760407,0.789766
600,0.405660,0.386538,0.820950,0.779856,0.814486
800,0.403131,0.372009,0.828518,0.772651,0.817498
1000,0.393610,0.404837,0.809275,0.781143,0.806071
1200,0.380029,0.359160,0.834084,0.772795,0.821063
1400,0.409019,0.356445,0.835296,0.792082,0.827859
1600,0.379195,0.354060,0.835246,0.793451,0.828212
1800,0.370721,0.399448,0.816077,0.787991,0.812792
2000,0.366901,0.344886,0.841628,0.789520,0.831287


Training Loss,Validation Loss,Step,Accuracy,F1,F1 Macro
0.329923,0.331984,4689,0.849493,0.807089,0.841852


... (3 outputs left out)

trainable params: 887,042 || all params: 125,534,212 || trainable%: 0.7066
A 147456 0.0001
B 147456 0.0008
other 592130 0.0001


Step,Training Loss,Validation Loss,Accuracy,F1,F1 Macro
200,0.448729,0.485907,0.766114,0.740733,0.763851
400,0.427731,0.402215,0.803463,0.768662,0.798912
600,0.405573,0.384834,0.822434,0.784097,0.816653
800,0.394535,0.368138,0.832154,0.777304,0.821314
1000,0.393483,0.409989,0.808088,0.781455,0.805195
1200,0.376209,0.356193,0.836186,0.775940,0.823419
1400,0.399158,0.361390,0.834776,0.797894,0.829084
1600,0.370512,0.349746,0.839871,0.798531,0.832833
1800,0.366850,0.393335,0.820134,0.792276,0.816839
2000,0.364343,0.340203,0.844942,0.793041,0.834536


Training Loss,Validation Loss,Step,Accuracy,F1,F1 Macro
0.321236,0.326467,4689,0.854514,0.811764,0.846602


Saved result: /content/LoRA-Plus/results/roberta/qqp/runs/roberta_qqp_lam8_lr0.0001_seed42_ep3_bs32_steps-1_screening.json
Batch 1 completed!


## 8B. LoRA+ Hyperparameter Screening - Batch 2

Evaluate the next two LoRA+ configurations using a base learning rate of 1e-4 with a ratio of 16× and a base learning rate of 2e-4 with a ratio of 4×.

Each configuration is trained for three epochs using random seed 42.

In [ ]:
# LoRA+ Screening — Batch 2
# Configurations 3 and 4

for lr, ratio in LORAPLUS_CONFIGS[2:4]:

    with track_compute(f"LoRA+ LR={lr}, ratio={ratio}"):

        print(f"\nRunning LoRA+: A LR={lr}, B LR={lr * ratio}, Ratio={ratio}")

        run_experiment(
            model_name=MODEL_NAME,
            dataset_name=DATASET_NAME,
            tokenizer=tokenizer,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            data_collator=data_collator,
            lam=ratio,
            seed=COMMON_CONFIG["screening_seed"],
            epochs=COMMON_CONFIG["epochs"],
            base_lr=lr,
            stage="screening"
        )

print("Batch 2 completed!")


Running LoRA+: A LR=0.0001, B LR=0.0016, Ratio=16


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 887,042 || all params: 125,534,212 || trainable%: 0.7066
A 147456 0.0001
B 147456 0.0016
other 592130 0.0001


Step,Training Loss,Validation Loss,Accuracy,F1,F1 Macro
200,0.443392,0.471971,0.778852,0.748034,0.775494
400,0.420224,0.428012,0.795993,0.768015,0.792982
600,0.416020,0.383076,0.821494,0.781601,0.815332


Step,Training Loss,Validation Loss,Accuracy,F1,F1 Macro
200,0.443392,0.471971,0.778852,0.748034,0.775494
400,0.420224,0.428012,0.795993,0.768015,0.792982
600,0.416020,0.383076,0.821494,0.781601,0.815332
800,0.405652,0.372270,0.832575,0.777986,0.821801


## 8C. LoRA+ Hyperparameter Screening - Batch 3

Evaluate the final two LoRA+ configurations using a base learning rate of 2e-4 and learning-rate ratios of 8× and 16×.

Each configuration is trained for three epochs using random seed 42.

In [ ]:
# LoRA+ Screening — Batch 3
# Configurations 5 and 6

for lr, ratio in LORAPLUS_CONFIGS[4:6]:

     with track_compute(f"LoRA+ LR={lr}, ratio={ratio}"):

        print(f"\nRunning LoRA+: A LR={lr}, B LR={lr * ratio}, Ratio={ratio}")

        run_experiment(
            model_name=MODEL_NAME,
            dataset_name=DATASET_NAME,
            tokenizer=tokenizer,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            data_collator=data_collator,
            lam=ratio,
            seed=COMMON_CONFIG["screening_seed"],
            epochs=COMMON_CONFIG["epochs"],
            base_lr=lr,
            stage="screening"
        )

print("Batch 3 completed!")

## 9. Best Hyperparameter Selection

Select the best-performing standard LoRA and LoRA+ configurations based on final validation binary F1 after three epochs.

Validation loss is used as a tie-breaker when configurations achieve identical F1 scores.

In [ ]:
best_lora, best_loraplus = select_best(
    MODEL_NAME,
    DATASET_NAME
)

BEST_LORA_LR = float(best_lora["eta_A"])
BEST_LORAPLUS_LR = float(best_loraplus["eta_A"])
BEST_LAM = int(best_loraplus["lam"])

print("\nSelected standard LoRA:")
print("A LR:", BEST_LORA_LR)
print("B LR:", BEST_LORA_LR)

print("\nSelected LoRA+:")
print("A LR:", BEST_LORAPLUS_LR)
print("B LR:", BEST_LORAPLUS_LR * BEST_LAM)
print("Ratio:", BEST_LAM)

## 10. Final Multi-Seed Experiments

Evaluate the selected standard LoRA and LoRA+ configurations across three random seeds: 42, 43, and 44.

Each run initializes a fresh pretrained model and trains for three epochs using the same dataset and experimental settings.

In [ ]:
for seed in COMMON_CONFIG["seeds"]:

    with track_compute(f"Final LoRA seed={seed}"):


        print(f"\nFinal LoRA run — seed {seed}")

        run_experiment(
            model_name=MODEL_NAME,
            dataset_name=DATASET_NAME,
            tokenizer=tokenizer,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            data_collator=data_collator,
            lam=1,
            seed=seed,
            epochs=COMMON_CONFIG["epochs"],
            base_lr=BEST_LORA_LR,
            stage="final"
        )

    with track_compute(f"Final LoRA+ seed={seed}"):

        print(f"\nFinal LoRA+ run — seed {seed}")

        run_experiment(
            model_name=MODEL_NAME,
            dataset_name=DATASET_NAME,
            tokenizer=tokenizer,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            data_collator=data_collator,
            lam=BEST_LAM,
            seed=seed,
            epochs=COMMON_CONFIG["epochs"],
            base_lr=BEST_LORAPLUS_LR,
            stage="final"
        )

## 11. Final Performance Summary

Aggregate the final experimental results across three random seeds.

Report mean and standard deviation for validation accuracy, binary F1, macro-F1, validation loss, training runtime, GPU memory usage, and trainable parameter count.

In [ ]:
summary = summarize_final(
    MODEL_NAME,
    DATASET_NAME
)

display(summary)

## 12. Convergence Analysis

Compare the convergence behavior of standard LoRA and LoRA+ using validation binary F1 against optimizer steps.

Plot the mean performance across three random seeds with standard-deviation bands to analyze training efficiency and convergence stability.

In [ ]:
plot_convergence(
    MODEL_NAME,
    DATASET_NAME
)

## 13. Save Experiment Results to GitHub

Verify, commit, and push completed experiment results to GitHub for persistent storage. Exclude model checkpoints from version control.

In [11]:
# Cell 13 — Verify, Commit, and Push Results to GitHub

import os
import subprocess
import tempfile
from pathlib import Path
from google.colab import userdata

PROJECT_DIR = Path("/content/LoRA-Plus")
root = get_output_dir(MODEL_NAME, DATASET_NAME)

# 1. Verify results
assert root.exists(), "Results directory not found"
assert (PROJECT_DIR / ".git").exists(), "Git repository not found"

print("Results directory:", root)

for path in sorted(root.rglob("*")):
    if path.is_file() and "checkpoints" not in path.parts:
        print(path.relative_to(root))

# 2. Configure Git identity
subprocess.run(
    ["git", "config", "user.name", "Ramprasath-Meganathan"],
    cwd=PROJECT_DIR,
    check=True
)

subprocess.run(
    ["git", "config", "user.email", "ramprasathmeganathan@gmail.com"],
    cwd=PROJECT_DIR,
    check=True
)

# 3. Stage results, excluding checkpoints
results_path = root.relative_to(PROJECT_DIR).as_posix()

subprocess.run(
    [
        "git", "add", "--all", "--",
        results_path,
        f":(exclude){results_path}/checkpoints/**"
    ],
    cwd=PROJECT_DIR,
    check=True
)

# 4. Commit only if changes exist
status = subprocess.run(
    ["git", "diff", "--cached", "--quiet"],
    cwd=PROJECT_DIR
)

if status.returncode == 1:
    subprocess.run(
        ["git", "commit", "-m",
         f"Save {MODEL_NAME} {DATASET_NAME} experiment results"],
        cwd=PROJECT_DIR,
        check=True
    )
elif status.returncode == 0:
    print("No new changes to commit.")
else:
    raise RuntimeError("Git status check failed")

# 5. Authenticate using Colab Secrets
token = userdata.get("GITHUB_TOKEN")

if not token:
    raise ValueError("GITHUB_TOKEN missing from Colab Secrets")

# 6. Push securely to GitHub
with tempfile.TemporaryDirectory() as temp_dir:

    askpass = Path(temp_dir) / "askpass.sh"

    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) echo "x-access-token" ;;\n'
        '  *Password*) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
        'esac\n'
    )

    askpass.chmod(0o700)

    env = os.environ.copy()
    env["GITHUB_TOKEN"] = token
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"

    subprocess.run(
        ["git", "push", "origin", "HEAD"],
        cwd=PROJECT_DIR,
        env=env,
        check=True
    )

print("Results successfully pushed to GitHub!")

Results directory: /content/LoRA-Plus/results/roberta/qqp
results_qqp.csv
runs/roberta_qqp_lam1_lr0.0001_seed42_ep3_bs32_steps-1_screening.json
runs/roberta_qqp_lam1_lr0.0002_seed42_ep3_bs32_steps-1_screening.json
runs/roberta_qqp_lam1_lr0.0004_seed42_ep3_bs32_steps-1_screening.json
runs/roberta_qqp_lam4_lr0.0001_seed42_ep3_bs32_steps-1_screening.json
runs/roberta_qqp_lam8_lr0.0001_seed42_ep3_bs32_steps-1_screening.json
No new changes to commit.


CalledProcessError: Command '['git', 'push', 'origin', 'HEAD']' returned non-zero exit status 128.